In [3]:

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Tue Mar 28 13:13:53 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.85.12    Driver Version: 525.85.12    CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   41C    P8    10W /  70W |      0MiB / 15360MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.nn.functional as functional
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score,f1_score
import time
import datetime

In [11]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [12]:
torch.manual_seed(0)
use_cuda = torch.cuda.is_available()
device = torch.device('cuda' if use_cuda else 'cpu')
if use_cuda:
    torch.cuda.manual_seed(0)
    
print("Using GPU: {}".format(use_cuda))

Using GPU: True


In [15]:
# Read the interventions datasest
import codecs
codecs.register_error('strict', codecs.lookup_error('surrogateescape'))
df = pd.read_csv('/content/drive/My Drive/Misinformation_Project/Bert_classified_impact_score_30K_interventions_21_22.csv') 
df.head()

,Unnamed: 0.4,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,User,Date Created,Tweet,Likes,Followers,...,Reply Effect,Retweet Effect,Individual Influence,Response Effect,Subjective Level,labels,Classify,Sign of Impact,Impact Value,Overall Impact
0,0,0,0,0,0,HSans,2022-12-30 23:42:46+00:00,"Here in #Qu??bec, #Canada we've had this rule ...",0,2335,...,0,0,3,0.0,3,0,M,1,4.500000,4.500000
1,1,1,1,1,1,Randi_RLB,2022-12-30 23:11:27+00:00,"@DrAGrace_cyhtt ""Alberta's education minister ...",0,1531,...,0,0,3,0.0,3,0,M,-1,3.577381,-3.577381
2,2,2,2,2,2,Golden_Pup,2022-12-30 22:51:45+00:00,November 18\n\nJust three days after defying a...,196,39775,...,1,2,4,2.0,1,1,C,-1,6.000000,-6.000000
3,3,3,3,3,3,EnigmaMachine8,2022-12-30 19:14:24+00:00,@shogun17761 @Katheri58113861 @kirkJK04 @jeanp...,0,2957,...,0,0,3,0.0,4,1,C,-1,4.000000,-4.000000
4,4,4,4,4,4,Cognisant2000,2022-12-30 18:59:47+00:00,@GMMalliet Agree. We always wear a mask in enc...,0,2037,...,0,0,3,0.0,1,0,M,-1,3.000000,-3.000000


In [16]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
# Define the independent variables set
X = df[['Likes', 'Followers', 'Friends','Reply','Retweet','Polarity','Subjectivity']]

# Create the VIF dataframe 
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns

# Calculate VIF for each feature
vif_data["VIF"] = [variance_inflation_factor(X.values,i) for i in range(len(X.columns))]

print(vif_data)

        feature       VIF
0         Likes  5.813654
1     Followers  1.063477
2       Friends  1.150960
3         Reply  1.796937
4       Retweet  7.230083
5      Polarity  1.045955
6  Subjectivity  1.187153
